# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR\u02b2 mass spectrometry analysis dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and its support for Croissant metadata.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed. Uncomment if running for the first time!
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is accessed as an object (not by subscripting)

# Display name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Review all record sets by their @id
record_sets = metadata.recordSet
if not record_sets:
    raise ValueError("No record sets found in metadata. Please check the schema definition.")
print("Record sets available:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else ''}")

# List the fields and columns for each record set by @id (using only the first record set for brevity)
# Update record_set_id based on printed output
record_set_id = record_sets[0]['@id']
rs = next(rs for rs in record_sets if rs['@id'] == record_set_id)
print(f"\nFields in record set {record_set_id}:")
for field in rs['field']:
    print(f"  - {field['@id']}: {field.get('name','')} (dataType: {field.get('dataType', '')})")

print(f"\nColumns in record set {record_set_id}:")
for col in rs['column']:
    print(f"  - {col['@id']}: {col.get('name','')} (source: {col.get('source','')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. The record set and field `@id`s are referenced as required.

In [ ]:
dataframes = {}

# If multiple record sets exist, you can loop through all; we use only the first for this demonstration.
selected_record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in selected_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet {rs_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for RecordSet {rs_id}: {e}")

# Use the first available record set for further examples
main_record_set_id = selected_record_set_ids[0]
main_df = dataframes[main_record_set_id]
print("Available columns:", main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on a numeric field, normalizing, and optionally grouping data. All references to fields are by their `@id`.

In [ ]:
# You should choose field @id for meaningful numeric fields, e.g. coverage percentage or MW (molecular weight).
# Replace '<numeric_field_id>' with the actual @id, as shown in previous output.
# If you do not know, run: print(main_df.columns.tolist())

# Example: let's use 'coverage_percentage' field, or fallback to the first numeric column
numeric_field = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break

if numeric_field is None:
    print("No numeric columns found. Please examine the DataFrame for numeric fields.")
filtered_df = main_df.copy()

# If a numeric field exists, continue with filtering and normalization
if numeric_field is not None:
    threshold = main_df[numeric_field].mean() if main_df[numeric_field].mean() is not None else 0
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical column (e.g. a sample ID, if available)
    group_field = None
    for col in main_df.columns:
        # Select a non-numeric field
        if not pd.api.types.is_numeric_dtype(main_df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (showing means of numeric fields):")
        print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions or relationships between fields by their `@id` using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If a second numeric column exists, plot correlation
    numeric_cols = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
    if len(numeric_cols) > 1:
        plt.figure(figsize=(7, 7))
        sns.scatterplot(data=main_df, x=numeric_cols[0], y=numeric_cols[1])
        plt.title(f"Scatter Plot of {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()

## 6. Conclusion
In this notebook, you loaded a Croissant-registered mass spectrometry dataset, explored its available record sets, fields, and columns using only the Croissant `@id` references, and performed simple processing and visualization. For further analysis, refer to the field definitions by their `@id`s in the provided metadata and consult the associated FAIR^2 documentation for the schema.
